# BF16 vs NVFP4 comparison

Reads what the harness already produced. Run the stages first:

```bash
python3 quantize.py --from download --through quality
```

This notebook renders the paired grid and the per-category breakdown. It computes nothing itself, so the numbers here are the same ones in `results/quality.json`.

**Before publishing or sharing any of this:** images generated from `flux-dev` are outputs of a non-commercially licensed model. Use the `flux-schnell` arm, which is Apache-2.0, for anything you intend to show or deploy.

In [ ]:
# SPDX-License-Identifier: Apache-2.0
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.

import json
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from common import paths

workspace = paths.resolve(os.environ.get("FLUX_QUANT_WORKSPACE"), create=False)
print(f"Workspace: {workspace.root}")
if workspace.ephemeral:
    print("warning: node-local workspace. Copy exports/ off before releasing the node.")

image_dir = workspace.images / "dynamic"
metadata = json.loads((image_dir / "metadata.json").read_text())
print(f"{len(metadata['images'])} images, {metadata['generation']['steps']} steps")

## Run provenance

Which GPU, which versions, which model revisions. Anyone reading the images should be able to see this without asking.

In [ ]:
manifest = json.loads((workspace.results / "run_manifest.json").read_text())
env = manifest["environment"]

print(f"host        {env['hostname']}  ({env['cpu_arch']})")
for device in env["gpu"].get("devices", []):
    label = {"10.0": "B200", "10.3": "GB300/B300"}.get(device["compute_capability"], "?")
    print(f"gpu         {device['name']}  cc {device['compute_capability']} ({label})  driver {device['driver']}")
for package, version in env["packages"].items():
    if version:
        print(f"{package:<12}{version}")

for stage in manifest["stages"]:
    print(f"  {stage['stage']:<10} {stage['status']:<8} {stage['duration_s']}s")

## Scores

CMMD is the primary distributional metric and PSNR a per-image signal. CLIP delta is the text-alignment proxy, and it is the one that maps to a named failure mode: a well-formed image with the wrong content is an automatic fail for them.

In [ ]:
quality_path = workspace.results / "quality.json"
if not quality_path.exists():
    print("No quality.json yet. Run: python3 quantize.py --stage quality")
else:
    quality = json.loads(quality_path.read_text())
    summary = quality["summary"]

    print(f"pairs            {summary['pairs']}")
    print(f"CMMD             {summary['cmmd'] if summary['cmmd'] is not None else 'not computed'}")
    print(f"PSNR median      {summary['psnr_db']['median']} dB   (min {summary['psnr_db']['min']})")
    print(f"  below 30 dB    {summary['psnr_db']['below_30db']}  - a signal, not a gate")
    print(f"CLIP delta mean  {summary['clip_delta']['mean']:+.3f}   worst {summary['clip_delta']['worst']:+.3f}")
    print("\nCLIP delta by category:")
    for category, delta in summary["clip_delta_by_category"].items():
        flag = "  <-- check" if delta < -0.5 else ""
        print(f"  {category:<20} {delta:+.3f}{flag}")

## Paired grid

Same prompt, same seed, same injected initial latents. The latent hash is printed under each row so the pairing can be verified rather than assumed.

This is the artefact their human reviewers would actually look at.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

prompts = {p["id"]: p for p in json.loads((Path.cwd().parent / "configs" / "prompts.json").read_text())["prompts"]}

pairs = {}
for record in metadata["images"]:
    pairs.setdefault((record["prompt_id"], record["seed"]), {})[record["arm"]] = record

complete = [(key, arms) for key, arms in sorted(pairs.items()) if len(arms) >= 2]
print(f"{len(complete)} complete pairs\n")

for (prompt_id, seed), arms in complete:
    arm_names = [a for a in ("bf16", "nvfp4-dynamic", "nvfp4-static") if a in arms]
    fig, axes = plt.subplots(1, len(arm_names), figsize=(6 * len(arm_names), 6.6))
    if len(arm_names) == 1:
        axes = [axes]

    for ax, arm in zip(axes, arm_names):
        ax.imshow(Image.open(image_dir / arms[arm]["file"]))
        ax.set_title(arm, fontsize=13)
        ax.axis("off")

    category = arms[arm_names[0]].get("category", "")
    digest = arms[arm_names[0]].get("latent_sha256_16", "")
    text = prompts.get(prompt_id, {}).get("text", "")
    fig.suptitle(f"{prompt_id}  ·  {category}  ·  seed {seed}  ·  latents {digest}", fontsize=11)
    fig.text(0.5, 0.02, text[:150], ha="center", fontsize=9, style="italic", wrap=True)
    plt.tight_layout(rect=[0, 0.05, 1, 0.96])
    plt.show()

## Worst cases first

Showing only the good pairs is how a comparison stops being evidence. Sort by CLIP delta and look at the bottom of the list.

In [ ]:
if quality_path.exists():
    # Loaded here too, so this cell runs standalone.
    quality = json.loads(quality_path.read_text())
    # A pair with no CLIP delta sorts as though it were the best and drops
    # out of a list whose whole purpose is to surface the worst. inf keeps
    # unscored pairs at the end, and a missing key no longer raises.
    worst = sorted(
        quality["pairs"],
        key=lambda p: p.get("clip_delta") if p.get("clip_delta") is not None else float("inf"),
    )[:5]
    print(f"{'prompt':<14}{'category':<18}{'seed':>6}{'PSNR':>9}{'CLIP delta':>12}")
    for entry in worst:
        psnr = entry.get('psnr_db')
        delta = entry.get('clip_delta')
        print(
            f"{entry['prompt_id']:<14}{str(entry.get('category')):<18}"
            f"{entry['seed']:>6}"
            + (f"{psnr:>9.1f}" if psnr is not None else f"{'--':>9}")
            + (f"{delta:>+12.3f}" if delta is not None else f"{'--':>12}")
        )
    print("\nInclude these in any write-up, not just the flattering pairs.")